In [2]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('../data/mental_health.sqlite')

df = pd.read_sql("""
    SELECT a.UserID, a.SurveyID, a.AnswerText, q.questiontext
    FROM Answer a
    JOIN Question q ON a.QuestionID = q.questionid
""", conn)

df_wide = df.pivot_table(
    index=['UserID', 'SurveyID'],
    columns='questiontext',
    values='AnswerText',
    aggfunc='first'
).reset_index()

In [3]:
threshold = 50
df_clean = df_wide.loc[:, df_wide.isnull().sum() / len(df_wide) * 100 < threshold]

print(df_clean.shape)
print(df_clean.columns.tolist())

(4218, 47)
['UserID', 'SurveyID', 'Are you self-employed?', 'Did your previous employers ever formally discuss mental health (as part of a wellness campaign or other official communication)?', 'Did your previous employers provide resources to learn more about mental health disorders and how to seek help?', 'Do you believe your productivity is ever affected by a mental health issue?', 'Do you currently have a mental health disorder?', 'Do you feel that your employer takes mental health as seriously as physical health?', 'Do you have a family history of mental illness?', 'Do you have medical coverage (private insurance or state-provided) that includes treatment of mental health disorders?', 'Do you have previous employers?', 'Do you know local or online resources to seek help for a mental health issue?', 'Do you know the options for mental health care available under your employer-provided health coverage?', 'Do you think that discussing a physical health issue with your employer would h

In [4]:
target = 'Have you ever sought treatment for a mental health disorder from a mental health professional?'

df_clean = df_clean.dropna(subset=[target])

print(df_clean.shape)
print(df_clean[target].value_counts())

(4218, 47)
Have you ever sought treatment for a mental health disorder from a mental health professional?
1    2412
0    1806
Name: count, dtype: int64


In [5]:
cols_to_drop = [
    'UserID',
    'If you live in the United States, which state or territory do you live in?',
    'What US state or territory do you work in?',
    'What country do you live in?',
    'What country do you work in?'
]

df_clean = df_clean.drop(columns=cols_to_drop)
print(df_clean.shape)

(4218, 42)


In [6]:
print(df_clean['What is your age?'].value_counts().head(20))

What is your age?
30    250
29    229
32    227
31    223
28    220
34    202
35    201
33    201
27    197
26    194
37    184
38    160
36    147
25    147
39    137
24    128
40    122
23    107
42    100
41     88
Name: count, dtype: int64


In [7]:
print(df_clean['What is your age?'].astype(float).describe())

count    4218.000000
mean       33.915363
std        10.478054
min       -29.000000
25%        28.000000
50%        33.000000
75%        38.000000
max       329.000000
Name: What is your age?, dtype: float64


In [8]:
df_clean['What is your age?'] = pd.to_numeric(df_clean['What is your age?'], errors='coerce')

df_clean = df_clean[df_clean['What is your age?'].between(18, 80)]

print(df_clean.shape)
print(df_clean['What is your age?'].describe())

(4202, 42)
count    4202.000000
mean       33.859829
std         8.065024
min        18.000000
25%        28.000000
50%        33.000000
75%        38.000000
max        74.000000
Name: What is your age?, dtype: float64


In [9]:
print(df_clean['What is your gender?'].value_counts())

What is your gender?
Male                            2821
Female                           914
male                             211
female                           110
-1                                22
                                ... 
43                                 1
masculino                          1
I am a Wookie                      1
Trans non-binary/genderfluid       1
Non-binary and gender fluid        1
Name: count, Length: 98, dtype: int64


In [10]:
def clean_gender(g):
    if pd.isna(g):
        return None
    g = str(g).strip().lower()
    if g in ['male', 'm', 'man', 'cis male', 'cis man', 'masculine', 
             'masculino', 'mail', 'malr', 'sex is male']:
        return 'Male'
    elif g in ['female', 'f', 'woman', 'cis female', 'cis woman', 
               'feminine', 'femail', 'femake', 'female (cis)']:
        return 'Female'
    else:
        return 'Other'

df_clean['What is your gender?'] = df_clean['What is your gender?'].apply(clean_gender)

print(df_clean['What is your gender?'].value_counts())

What is your gender?
Male      3035
Female    1024
Other      143
Name: count, dtype: int64


In [11]:
missing = (df_clean.isnull().sum() / len(df_clean) * 100).sort_values(ascending=False)
print(missing[missing > 0])

questiontext
Do you feel that your employer takes mental health as seriously as physical health?                                                                                                 36.220847
Do you think that discussing a physical health issue with your employer would have negative consequences?                                                                           36.220847
Do you currently have a mental health disorder?                                                                                                                                     29.795336
Do you have medical coverage (private insurance or state-provided) that includes treatment of mental health disorders?                                                              29.795336
Do you believe your productivity is ever affected by a mental health issue?                                                                                                         29.795336
Did your previous employers provide r

In [12]:
df_model = df_clean.dropna()
print(df_model.shape)
print(df_model[target].value_counts())

(1428, 42)
Have you ever sought treatment for a mental health disorder from a mental health professional?
1    837
0    591
Name: count, dtype: int64
